# EDA

In [1]:
import os
import pandas as pd

# Define paths based on your directory structure
LABEL_DIR = "./mrnet_images/lable"

# Load Training CSVs
train_abnormal = pd.read_csv(os.path.join(LABEL_DIR, 'train_abnormal.csv'), header=None, names=['exam_id', 'abnormal'], dtype={'exam_id': str})
train_acl = pd.read_csv(os.path.join(LABEL_DIR, 'train_acl.csv'), header=None, names=['exam_id', 'acl'], dtype={'exam_id': str})
train_meniscus = pd.read_csv(os.path.join(LABEL_DIR, 'train_meniscus.csv'), header=None, names=['exam_id', 'meniscus'], dtype={'exam_id': str})

# Load Validation CSVs
valid_abnormal = pd.read_csv(os.path.join(LABEL_DIR, 'valid_abnormal.csv'), header=None, names=['exam_id', 'abnormal'], dtype={'exam_id': str})
valid_acl = pd.read_csv(os.path.join(LABEL_DIR, 'valid_acl.csv'), header=None, names=['exam_id', 'acl'], dtype={'exam_id': str})
valid_meniscus = pd.read_csv(os.path.join(LABEL_DIR, 'valid_meniscus.csv'), header=None, names=['exam_id', 'meniscus'], dtype={'exam_id': str})

# ID Alignment Verification using set mathematics
train_ids_match = set(train_abnormal['exam_id']) == set(train_acl['exam_id']) == set(train_meniscus['exam_id'])
valid_ids_match = set(valid_abnormal['exam_id']) == set(valid_acl['exam_id']) == set(valid_meniscus['exam_id'])

print(f"All Training Exam IDs align perfectly: {train_ids_match}")
print(f"All Validation Exam IDs align perfectly: {valid_ids_match}")

All Training Exam IDs align perfectly: True
All Validation Exam IDs align perfectly: True


In [2]:
# Merge training categories
train_df = train_abnormal.merge(train_acl, on='exam_id').merge(train_meniscus, on='exam_id')
train_df['split'] = 'train'

# Merge validation categories
valid_df = valid_abnormal.merge(valid_acl, on='exam_id').merge(valid_meniscus, on='exam_id')
valid_df['split'] = 'valid'

# Combine into one master dataframe
master_labels_df = pd.concat([train_df, valid_df], ignore_index=True)
print(f"Master label shape: {master_labels_df.shape}")

Master label shape: (1250, 5)


In [10]:
# 1. Drop the rows containing underscores
master_labels_clean = master_labels_df[~master_labels_df['exam_id'].str.contains('_', na=False)].copy()

# 2. Explicitly cast the columns to integers
target_columns = ['abnormal', 'acl', 'meniscus']
for col in target_columns:
    master_labels_clean[col] = master_labels_clean[col].astype(int)

# 3. Print verification metrics
print(f"Cleaned Master Label Shape: {master_labels_clean.shape} (Expected: (1248, 5))")
print("\nData Types After Cleaning:")
print(master_labels_clean.dtypes)
print("\nFirst 3 rows of cleaned data:")
print(master_labels_clean.head(10))

Cleaned Master Label Shape: (1248, 5) (Expected: (1248, 5))

Data Types After Cleaning:
exam_id       str
abnormal    int64
acl         int64
meniscus    int64
split         str
dtype: object

First 3 rows of cleaned data:
   exam_id  abnormal  acl  meniscus  split
1     1027         0    0         0  train
2     0228         0    0         0  train
3     1053         1    0         1  train
4     1069         1    0         0  train
5     0278         1    0         0  train
6     0567         1    0         0  train
7     0087         1    1         0  train
8     0927         1    1         1  train
9     0964         1    1         1  train
10    0686         1    0         0  train


In [15]:
# Define your local root path where 'axial', 'coronal', 'sagittal' folders live
DATA_ROOT = "./mrnet_images"

missing_files = []

for _, row in master_labels_clean.iterrows():
    exam_id = row['exam_id']
    split = row['split']
    
    # Determine directory path based on your local folder layout
    if split == 'train':
        # Paths like: ./mrnet_images/axial/1027.npy
        orientations = {
            'axial': os.path.join(DATA_ROOT, 'axial', f'{exam_id}.npy'),
            'coronal': os.path.join(DATA_ROOT, 'coronal', f'{exam_id}.npy'),
            'sagittal': os.path.join(DATA_ROOT, 'sagittal', f'{exam_id}.npy')
        }
    else:
        # Paths like: ./mrnet_images/valid_files/axial/xxxx.npy
        orientations = {
            'axial': os.path.join(DATA_ROOT, 'valid_files', 'axial', f'{exam_id}.npy'),
            'coronal': os.path.join(DATA_ROOT, 'valid_files', 'coronal', f'{exam_id}.npy'),
            'sagittal': os.path.join(DATA_ROOT, 'valid_files', 'sagittal', f'{exam_id}.npy')
        }
    
    # Check if all three structural views exist on disk
    for view, path in orientations.items():
        if not os.path.exists(path):
            missing_files.append({'exam_id': exam_id, 'split': split, 'view': view, 'expected_path': path})

# Summary of the audit
if len(missing_files) == 0:
    print("Disk Audit Passed! All 1,248 exams have complete Axial, Coronal, and Sagittal files.")
else:
    print(f"[WARNING] Audit Failed. Missing {len(missing_files)} total files.")
    missing_df = pd.DataFrame(missing_files)
    print(missing_df.head())

[WARNING] Audit Failed. Missing 187 total files.
  exam_id  split      view                     expected_path
0    0964  train  sagittal  ./mrnet_images/sagittal/0964.npy
1    0949  train  sagittal  ./mrnet_images/sagittal/0949.npy
2    0743  train  sagittal  ./mrnet_images/sagittal/0743.npy
3    0045  train  sagittal  ./mrnet_images/sagittal/0045.npy
4    0398  train     axial     ./mrnet_images/axial/0398.npy


In [16]:
# 1. Look at the distribution of the missing files
print("--- Missing Files Distribution ---")
print(missing_df.groupby(['split', 'view']).size())

# 2. Inspect the actual file names on your disk to check for padding issues
print("\n--- Inspecting Physical Disk Filenames ---")
folders_to_check = [
    "./mrnet_images/sagittal",
    "./mrnet_images/axial"
]

for folder in folders_to_check:
    if os.path.exists(folder):
        # Grab the first 10 files physically present in that directory
        actual_files = sorted([f for f in os.listdir(folder) if f.endswith('.npy')])
        print(f"\nFirst 10 files physically present in '{folder}':")
        print(actual_files[:10])
    else:
        print(f"\nFolder not found: {folder}")

--- Missing Files Distribution ---
split  view    
train  axial       47
       coronal     47
       sagittal    93
dtype: int64

--- Inspecting Physical Disk Filenames ---

First 10 files physically present in './mrnet_images/sagittal':
['0001.npy', '0002.npy', '0003.npy', '0004.npy', '0006.npy', '0007.npy', '0009.npy', '0013.npy', '0014.npy', '0015.npy']

First 10 files physically present in './mrnet_images/axial':
['0000.npy', '0001.npy', '0002.npy', '0003.npy', '0004.npy', '0005.npy', '0006.npy', '0007.npy', '0008.npy', '0009.npy']


## Notes:
there is some missing files we see asystematic missingness between axial and coronal but more in sagittal view. Which means we need to count for those missing files.

In [17]:
# 1. Extract the unique exam IDs that are missing at least one .npy file
missing_ids = set(missing_df['exam_id'].unique())

# 2. Filter out these incomplete exams from our clean labels dataframe
final_labels_df = master_labels_clean[~master_labels_clean['exam_id'].isin(missing_ids)].copy()

# 3. Print dataset reduction metrics
print("--- Dataset Filtering Summary ---")
print(f"Exams in labels files: {len(master_labels_clean)}")
print(f"Exams with missing files dropped: {len(missing_ids)}")
print(f"Final valid exams for local modeling: {len(final_labels_df)}")
print(f"Dropped {len(master_labels_clean) - len(final_labels_df)} rows from the labels matrix.")

# 4. Compute class balances across both splits
print("\n--- Class Balance Breakdown ---")
for split_name in ['train', 'valid']:
    split_subset = final_labels_df[final_labels_df['split'] == split_name]
    total_cases = len(split_subset)
    
    print(f"\n================ {split_name.upper()} SPLIT (Total: {total_cases} Exams) ================")
    
    for condition in ['abnormal', 'acl', 'meniscus']:
        counts = split_subset[condition].value_counts()
        percentages = split_subset[condition].value_counts(normalize=True) * 100
        
        print(f"\nCondition: {condition.upper()}")
        for value in [0, 1]:
            count = counts.get(value, 0)
            pct = percentages.get(value, 0.0)
            status = "Positive (1)" if value == 1 else "Negative (0)"
            print(f"  {status}: {count} cases ({pct:.2f}%)")

--- Dataset Filtering Summary ---
Exams in labels files: 1248
Exams with missing files dropped: 186
Final valid exams for local modeling: 1062
Dropped 186 rows from the labels matrix.

--- Class Balance Breakdown ---

================ TRAIN SPLIT (Total: 943 Exams) ================

Condition: ABNORMAL
  Negative (0): 184 cases (19.51%)
  Positive (1): 759 cases (80.49%)

Condition: ACL
  Negative (0): 757 cases (80.28%)
  Positive (1): 186 cases (19.72%)

Condition: MENISCUS
  Negative (0): 610 cases (64.69%)
  Positive (1): 333 cases (35.31%)

================ VALID SPLIT (Total: 119 Exams) ================

Condition: ABNORMAL
  Negative (0): 24 cases (20.17%)
  Positive (1): 95 cases (79.83%)

Condition: ACL
  Negative (0): 65 cases (54.62%)
  Positive (1): 54 cases (45.38%)

Condition: MENISCUS
  Negative (0): 67 cases (56.30%)
  Positive (1): 52 cases (43.70%)


In [18]:
final_labels_df = master_labels_clean[~master_labels_clean['exam_id'].isin(missing_ids)]

### Notes on the final label data set and the balance issue:
After isolating incomplete exams we are left with a perfectly clean dataset of 1,062 total valid exams (943 for training, 119 for validation).

### Class balance breakdown:
**1- The Training Split Skew (943 Exams):**

- ***Abnormal (80.49% Positive):*** The vast majority of patients sent to get an MRI at Stanford actually have something wrong with their knee. Only ~20% of your training data represents a completely normal knee.

- ***ACL Tears (19.72% Positive):*** This is highly skewed. Roughly 4 out of 5 scans do not have an ACL tear. This means our baseline model cannot use simple accuracy as its success metric—if it blindly guesses "No Tear" every time, it will achieve 80% accuracy while being completely useless at finding injuries.

- ***Meniscus Tears (35.31% Positive):*** A much healthier, moderate distribution.

**The Validation Split Surprise (119 Exams)**
Look closely at your validation split numbers:

- ACL Positive: 45.38%

- Meniscus Positive: 43.70%

The validation split is significantly more balanced than the training split. This is a deliberate design choice by the Stanford dataset creators. They intentionally enriched the validation set with more injury cases so that evaluating the model's diagnostic performance would be statistically robust.

In [19]:
# Save the filtered DataFrame as our project's static data manifest
output_path = "./unified_labels.csv"

# We keep index=True so that 'exam_id' remains structured as a primary column or index
final_labels_df.to_csv(output_path)

print(f"Successfully exported {len(final_labels_df)} verified exam rows to '{output_path}'.")
print("This CSV will now serve as the single source of truth for both your local venv and Colab.")

Successfully exported 1062 verified exam rows to './unified_labels.csv'.
This CSV will now serve as the single source of truth for both your local venv and Colab.
